# What the loop does when things go wrong

`01_one_check.ipynb` walks the happy path with a real container. This notebook
is about behaviour, so it uses a stand-in for the job — each case runs in
seconds, and the awkward situations can be staged deliberately. Everything else
is real: same database, same storage, same loop code.

Each case states its setup, its expectation, then shows what happened.

In [1]:
import json

import pandas as pd

from recon import check, db, gap, identity, intent, jobs, observe, processing, queue, storage
from recon.workers import JobStatus

s3 = storage.get_s3_client()
BUCKET, _ = storage.parse_s3_path(storage.model_base_path(0))
DOMAIN = "N10S10E10W10"          # the domain code a build would emit
LAKE = "testlake"


class StandInRunner:
    """Accepts submissions and reports whatever status a case needs.

    It writes no artifacts — cases place those in storage themselves, which is
    the honest way to test a loop that believes storage over jobs.
    """

    def __init__(self):
        self.submitted, self.status = [], JobStatus.RUNNING

    def submit(self, job, payload):
        # Each job names the reach its own way: build_model takes it directly,
        # run_nd_scenarios only ever sees paths. Both are addresses the loop
        # built, so either way the reach is recoverable from the payload.
        reach_id = payload.get("reach_id") or int(
            payload["model_manifest_path"].split("/reach=")[1].split("/")[0])
        self.submitted.append((job, reach_id))
        return f"stand-in-{reach_id}-{len(self.submitted)}"

    def poll(self, ref):
        return self.status

    def reap(self, ref):
        pass

    def logs(self, ref, tail=50):
        return "stand-in failure"


LULC = {"11": 0.04, "21": 0.04}

def reset(*reach_ids, water_body=True):
    """Terminal reaches with no dependencies, defaults seeded, storage clean.

    Terminal because this notebook is about one reach at a time; the cascade is
    `03_run_network.ipynb`'s story. `water_body=False` stages the one terminal
    that cannot run ND at all — it drains nowhere the system knows about.
    """
    with db.connect() as conn:
        conn.execute("TRUNCATE reach_network, lakes CASCADE")
        conn.execute("DELETE FROM desired_state_defaults")
        conn.execute(
            """INSERT INTO desired_state_defaults
               (sdr_commit, grid_resolution, epsg_code, dem_source, lulc_source, lulc_lookup,
                solver, solver_version, q_lower_bound, q_upper_bound, initial_dq_step_for_nd)
               VALUES ('deadbeefcafe', 10, 5070, 's3://dem', 's3://lulc', %s,
                       'lisflood', '8.1.0', 10, 100, 10)""",
            (json.dumps(LULC),))
        conn.execute(
            "INSERT INTO lakes (lake_id, geom) VALUES (%s, "
            "ST_GeomFromText('MULTIPOLYGON(((0 0,0 1,1 1,1 0,0 0)))', 5070))", (LAKE,))
        for rid in reach_ids:
            wkt = f"LINESTRING(0 0,{rid} 1)"  # distinct geometry -> distinct identity per reach
            conn.execute(
                "INSERT INTO reach_network (reach_id, is_terminal, terminal_reason, lake_to_id,"
                " slope, geom) VALUES (%s, TRUE, %s, %s, 0.001, ST_GeomFromText(%s, 5070))",
                (rid, "lake" if water_body else "outlet", LAKE if water_body else None, wkt))
        conn.execute("INSERT INTO desired_state (reach_id) SELECT reach_id FROM reach_network")
    # The lake's outflow polygon is published once per water body, not per reach.
    s3.put_object(Bucket=BUCKET,
                  Key=storage.parse_s3_path(storage.boundary_polygon_path("lake", LAKE))[1],
                  Body=json.dumps({"type": "FeatureCollection", "features": []}).encode())
    for rid in reach_ids:
        clear_storage(rid)
    return StandInRunner()


def clear_storage(reach_id, models=True, runs=True):
    """Remove a reach's artifacts. Models and runs separately, so a case can
    delete a library without disturbing the model it was run against."""
    prefixes = []
    if models:
        prefixes.append(storage.parse_s3_path(storage.model_base_path(reach_id))[1])
    if runs:
        prefixes.append(f"version=v1/results/reach={reach_id}")
    for prefix in prefixes:
        for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix).get("Contents", []):
            s3.delete_object(Bucket=BUCKET, Key=obj["Key"])


def put_model(reach_id, manifest=True, break_identity=False):
    """Stage what a finished build leaves behind — at the PREDICTED address,
    with a manifest that passes verification (unless asked to break it)."""
    wanted = intent.effective(reach_id)
    identity_obj, ihash = identity.model_identity(wanted)
    if break_identity:
        identity_obj = {**identity_obj, "grid_resolution": 999.0}  # hash no longer matches
    model_id = f"{ihash}_{DOMAIN}"
    _, base = storage.parse_s3_path(storage.model_base_path(reach_id))
    s3.put_object(Bucket=BUCKET, Key=f"{base}/{model_id}/dem.tif", Body=b"raster")
    if manifest:
        s3.put_object(
            Bucket=BUCKET, Key=f"{base}/{model_id}/{storage.MANIFEST_FILENAME}",
            Body=json.dumps({"reach_id": reach_id, "identity_hash": ihash,
                             "identity": identity_obj, "model_id": model_id,
                             "created_at": "2026-08-19T00:00:00Z"}).encode())
    return model_id


def put_nd_library(reach_id, discharges=None, manifest=True):
    """Stage a normal-depth library at the predicted address.

    `discharges` defaults to a set that spans the authored range. Pass a
    narrower one to stage a library that does not satisfy intent.
    """
    wanted = intent.effective(reach_id)
    _, mhash = identity.model_identity(wanted)
    run_obj, rhash = identity.run_identity(wanted)
    model_id = f"{mhash}_{DOMAIN}"
    library = storage.nd_library_path(reach_id, mhash, rhash, wanted["slope"])
    _, base = storage.parse_s3_path(library)
    if discharges is None:
        discharges = [wanted["q_lower_bound"], 55, wanted["q_upper_bound"]]
    for q in discharges:
        folder = f"{base}/{identity.q_folder(q)}"
        # The inundated area is what the reach ABOVE will drain through.
        s3.put_object(Bucket=BUCKET, Key=f"{folder}/{storage.INUNDATED_AREA_FILENAME}",
                      Body=b'{"type":"FeatureCollection","features":[]}')
        if manifest:
            s3.put_object(
                Bucket=BUCKET, Key=f"{folder}/{storage.SCENARIO_MANIFEST_FILENAME}",
                Body=json.dumps({"reach_id": reach_id, "identity": run_obj,
                                 "identity_hash": rhash, "model_id": model_id,
                                 "inputs": {"us_discharge": float(q)},
                                 "properties": {"nominal_wse": 200.0 + q / 10}}).encode())
    return library


def put_everything(reach_id):
    """A reach whose model and library both already exist."""
    put_model(reach_id)
    put_nd_library(reach_id)


def state_of(reach_id):
    return db.one("SELECT state, model_id, nd_materialized, nd_discharges, "
                  "model_applied_revision, nd_applied_revision, desired_revision, has_gap "
                  "FROM reach_status WHERE reach_id = %s", (reach_id,))


print("ready")

ready


## Case 1 — what already exists is adopted, not rebuilt

**Setup:** a model *and* its normal-depth library already sit at the addresses
intent implies (an earlier deployment, a restored bucket, someone's `aws s3 sync`).
**Expect:** the first check ever run adopts both and the reach is finished —
no job, no container, nothing to clean up.

In [2]:
runner = reset(1)
put_everything(1)

print("check:", check.run_check(1, runner))
print("state:", state_of(1))
print("jobs submitted:", runner.submitted)

check: reach 1 | rev 0 | NoGap | satisfied
state: {'state': 'finished', 'model_id': 'e18dcada_N10S10E10W10', 'nd_materialized': True, 'nd_discharges': 3, 'model_applied_revision': 0, 'nd_applied_revision': 0, 'desired_revision': 0, 'has_gap': False}
jobs submitted: []


## Case 2 — the ladder climbs one rung per check, and nothing is submitted twice

**Setup:** one reach, nothing built.
**Expect:** a check submits `build_model` and a second check leaves it alone.
When the model appears, the *same* check that adopts it moves to the next rung
and submits `run_nd_scenarios` — the reach is not finished just because its
model is. When the library appears too, the reach is done and further checks
are no-ops.

Two submissions across five checks, and each one only after the rung below it
was proved by looking at storage.

In [3]:
runner = reset(2)

print("check 1:", check.run_check(2, runner))
print("check 2:", check.run_check(2, runner), "  <- already in flight")

runner.status = JobStatus.SUCCEEDED
put_model(2)                                  # the build's output appears
jobs.status_pass(runner)
print("check 3:", check.run_check(2, runner), "  <- model adopted, nd submitted")

put_nd_library(2)                             # the library's output appears
jobs.status_pass(runner)
print("check 4:", check.run_check(2, runner))
print("check 5:", check.run_check(2, runner))

print(f"\nsubmissions: {runner.submitted}")
print("state:      ", state_of(2))
print("still due:  ", [r["reach_id"] for r in queue.due_reaches()])

check 1: reach 2 | rev 0 | RunStep | submitted build_model (stand-in-2-1)
check 2: reach 2 | rev 0 | InFlight | build_model already running, left alone   <- already in flight


check 3: reach 2 | rev 0 | RunStep | submitted run_nd_scenarios (stand-in-2-2)   <- model adopted, nd submitted


check 4: reach 2 | rev 0 | NoGap | satisfied


check 5: reach 2 | rev 0 | NoGap | satisfied

submissions: [('build_model', 2), ('run_nd_scenarios', 2)]
state:       {'state': 'finished', 'model_id': '2d92bb9b_N10S10E10W10', 'nd_materialized': True, 'nd_discharges': 3, 'model_applied_revision': 0, 'nd_applied_revision': 0, 'desired_revision': 0, 'has_gap': False}
still due:   []


## Case 3 — a half-written model does not count

**Setup:** artifacts in storage, but no manifest — a build that died partway.
**Expect:** invisible. `build_model` writes `model_manifest.json` last, so the
manifest's presence is the only honest signal a build finished.

In [4]:
runner = reset(3)
put_model(3, manifest=False)

print("observe:", observe.observe_reach(3))
print("check:  ", check.run_check(3, runner))

observe: {'reach_id': 3, 'predicted': '3bf9d100', 'found': None, 'changed': False, 'refused': [], 'was': None}


check:   reach 3 | rev 0 | RunStep | submitted build_model (stand-in-3-1)


## Case 4 — a manifest that lies is refused

**Setup:** a manifest at the right address whose identity object does not hash
to the identity it claims — a hand-edited file, a corrupted upload, or drift
between the loop's hashing recipe and the job's.
**Expect:** not adopted. The refusal is loud and the reach is treated as
unbuilt, because adopting a hash we cannot reproduce would mean trusting a
label over the contents.

In [5]:
runner = reset(4)
put_model(4, break_identity=True)

seen = observe.observe_reach(4)
print("adopted:", seen["found"])
print("refused:", json.dumps(seen["refused"], indent=2))
print("check:  ", check.run_check(4, runner))

refused manifest at s3://twod-fim-artifacts/version=v1/models/reach=4/2b7c38c2_N10S10E10W10: ["identity object hashes to 37952076, manifest claims 2b7c38c2: the hashing recipe here has drifted from the job's"]


refused manifest at s3://twod-fim-artifacts/version=v1/models/reach=4/2b7c38c2_N10S10E10W10: ["identity object hashes to 37952076, manifest claims 2b7c38c2: the hashing recipe here has drifted from the job's"]


adopted: None
refused: [
  {
    "folder": "2b7c38c2_N10S10E10W10",
    "problems": [
      "identity object hashes to 37952076, manifest claims 2b7c38c2: the hashing recipe here has drifted from the job's"
    ]
  }
]


check:   reach 4 | rev 0 | RunStep | submitted build_model (stand-in-4-1)


## Case 5 — a partial library is not a smaller proof, it is none

**Setup:** a model, and a library that stops at 40 cms when intent asks for
10–100. This is what a run job looks like halfway through, or after someone
deleted the scenarios they thought were spare.
**Expect:** no row at all. A `materialized_*` row means *this step's desired
state is materialized*; there is no way to write "partly". The reach asks for
the library again, and re-running it is cheap because the scenarios that do
exist are content-addressed and returned early.

In [6]:
runner = reset(5)
put_model(5)
put_nd_library(5, discharges=[10, 40])        # stops short of the authored 100
observe.observe_reach(5)                      # adopt the model, so the runs have an address

seen = observe.observe_nd_runs(5)
print("adopted:", seen["found"], "|", seen["note"])
print("check:  ", check.run_check(5, runner))

put_nd_library(5)                             # now the full span
print("\nonce it spans:", observe.observe_nd_runs(5)["q_set"])
print("check:  ", check.run_check(5, runner))
print("state:  ", state_of(5))

adopted: None | library spans 10-40, intent asks for 10-100


check:   reach 5 | rev 0 | RunStep | submitted run_nd_scenarios (stand-in-5-1)

once it spans: [10, 40, 55, 100]


check:   reach 5 | rev 0 | NoGap | satisfied
state:   {'state': 'finished', 'model_id': '54d125f0_N10S10E10W10', 'nd_materialized': True, 'nd_discharges': 4, 'model_applied_revision': 0, 'nd_applied_revision': 0, 'desired_revision': 0, 'has_gap': False}


## Case 6 — deleting from storage is how you undo, and it cascades

**Setup:** a finished reach; then its **model** is deleted from the bucket.
**Expect:** the next check finds nothing at the model address and deletes that
proof row. The library's proof goes too — not because anything tracked a
dependency, but because a run is addressed *underneath* the model it was run
against, so with no materialized model there is nowhere for the loop to look.
Retraction falls out of addressing rather than bookkeeping.

In [7]:
runner = reset(6)
put_everything(6)
check.run_check(6, runner)
print("before:", state_of(6))

clear_storage(6, runs=False)                  # delete the model, leave the runs
queue.request_check(6)

print("after: ", check.run_check(6, runner))
print("state: ", state_of(6))
print("nd row:", db.one("SELECT count(*) AS rows FROM materialized_nd_runs WHERE reach_id = 6"))

before: {'state': 'finished', 'model_id': 'e6ef5e9f_N10S10E10W10', 'nd_materialized': True, 'nd_discharges': 3, 'model_applied_revision': 0, 'nd_applied_revision': 0, 'desired_revision': 0, 'has_gap': False}


after:  reach 6 | rev 0 | RunStep | submitted build_model (stand-in-6-1)
state:  {'state': 'in_flight', 'model_id': None, 'nd_materialized': False, 'nd_discharges': None, 'model_applied_revision': -1, 'nd_applied_revision': -1, 'desired_revision': 0, 'has_gap': True}
nd row: {'rows': 0}


## Case 7 — intent changes reopen the gap; reverting re-adopts for free

**Setup:** a finished reach. Then the deployment's `grid_resolution` changes.
**Expect:** every reach's revision bumps, the predicted model address moves,
and the old model — still in the bucket — no longer counts. Reverting makes the
*original* address correct again, and the next check adopts both the model and
its library back **without running any job**.

This is gap kind 4, and it needed no staleness machinery — the address moved,
that is all.

In [8]:
runner = reset(7)
put_everything(7)
check.run_check(7, runner)
print("satisfied:      ", state_of(7))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET grid_resolution = 30")
print("intent changed: ", check.run_check(7, runner), "| submissions:", len(runner.submitted))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET grid_resolution = 10")
print("reverted:       ", check.run_check(7, runner), "| submissions:", len(runner.submitted))
print("state:          ", state_of(7))

satisfied:       {'state': 'finished', 'model_id': '00a2bda8_N10S10E10W10', 'nd_materialized': True, 'nd_discharges': 3, 'model_applied_revision': 0, 'nd_applied_revision': 0, 'desired_revision': 0, 'has_gap': False}
intent changed:  reach 7 | rev 1 | RunStep | submitted build_model (stand-in-7-1) | submissions: 1


reverted:        reach 7 | rev 2 | NoGap | satisfied | submissions: 1
state:           {'state': 'finished', 'model_id': '00a2bda8_N10S10E10W10', 'nd_materialized': True, 'nd_discharges': 3, 'model_applied_revision': 2, 'nd_applied_revision': 2, 'desired_revision': 2, 'has_gap': False}


## Case 8 — a solver upgrade moves the runs but not the model

**Setup:** a finished reach, then `solver_version` moves to 8.2.0.
**Expect:** the model is untouched — the solver is no part of model identity —
but the run address moves, so the library must be produced again. This is the
Identity/Realization split doing its job: the expensive terrain work survives a
solver upgrade, and only the runs are redone.

In [9]:
runner = reset(8)
put_everything(8)
check.run_check(8, runner)
print("before:", state_of(8))

with db.connect() as conn:
    conn.execute("UPDATE desired_state_defaults SET solver_version = '8.2.0'")

print("after: ", check.run_check(8, runner))
print("state: ", state_of(8), "  <- model still proved, nd is not")
print("submitted:", runner.submitted)

before: {'state': 'finished', 'model_id': '96229c3b_N10S10E10W10', 'nd_materialized': True, 'nd_discharges': 3, 'model_applied_revision': 0, 'nd_applied_revision': 0, 'desired_revision': 0, 'has_gap': False}


after:  reach 8 | rev 1 | RunStep | submitted run_nd_scenarios (stand-in-8-1)
state:  {'state': 'in_flight', 'model_id': '96229c3b_N10S10E10W10', 'nd_materialized': False, 'nd_discharges': None, 'model_applied_revision': 1, 'nd_applied_revision': -1, 'desired_revision': 1, 'has_gap': True}   <- model still proved, nd is not
submitted: [('run_nd_scenarios', 8)]


## Case 9 — a terminal that drains nowhere awaits inputs, not retries

**Setup:** a terminal reach with no lake and no coast — a clip edge, or an
outlet nobody has classified.
**Expect:** its model builds, and then it stops. The normal-depth boundary is
the polygon of the water body a terminal drains into, and no job can invent
one. So the loop reports `AwaitingInputs` rather than submitting work that must fail:
a failure would burn the retry budget and end at `halted` with a misleading
error, when the truth is that someone needs to author data.

In [10]:
runner = reset(9, water_body=False)
put_model(9)

print("check:", check.run_check(9, runner))
print("state:", state_of(9))
print("submissions:", runner.submitted, "| failures:",
      db.one("SELECT consecutive_failures, halted FROM reach_processing WHERE reach_id = 9"))

check: reach 9 | rev 0 | AwaitingInputs | run_nd_scenarios awaiting inputs: terminal reach names no lake or coast to drain into
state: {'state': 'awaiting_inputs', 'model_id': '51499d6e_N10S10E10W10', 'nd_materialized': False, 'nd_discharges': None, 'model_applied_revision': 0, 'nd_applied_revision': -1, 'desired_revision': 0, 'has_gap': True}
submissions: [] | failures: {'consecutive_failures': 0, 'halted': False}


## Case 10 — failures back off, then stop

**Setup:** a runner that cannot submit at all.
**Expect:** each failure counted, the wait doubling, then the reach parked
(`halted`) for a person. Retries belong to the loop — nothing else needs to
be configured to retry, and nothing else should be.

In [11]:
class BrokenRunner(StandInRunner):
    def submit(self, job, payload):
        raise RuntimeError("docker daemon unreachable")

reset(10)
broken = BrokenRunner()
rows = []
for attempt in range(6):
    check.run_check(10, broken)
    row = db.one("SELECT consecutive_failures, halted FROM reach_processing WHERE reach_id = 10")
    rows.append({"attempt": attempt + 1, **row,
                 "due": any(r["reach_id"] == 10 for r in queue.due_reaches())})
    with db.connect() as conn:   # skip the wait so the case finishes quickly
        conn.execute("UPDATE reach_processing SET next_retry_at = NULL WHERE reach_id = 10")

display(pd.DataFrame(rows))
processing.clear_halt(10)
print("after clear_halt, due:", [r["reach_id"] for r in queue.due_reaches()])

check failed for reach 10
Traceback (most recent call last):
  File "/home/ert/dev/git-repos/fim/2d-fim/twod-fim-deployment/orchestrator/recon/check.py", line 270, in run_check
    ref = runner.submit(decision.step, PAYLOADS[decision.step](reach_id))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3266158/4023097037.py", line 3, in submit
    raise RuntimeError("docker daemon unreachable")
RuntimeError: docker daemon unreachable


check failed for reach 10
Traceback (most recent call last):
  File "/home/ert/dev/git-repos/fim/2d-fim/twod-fim-deployment/orchestrator/recon/check.py", line 270, in run_check
    ref = runner.submit(decision.step, PAYLOADS[decision.step](reach_id))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3266158/4023097037.py", line 3, in submit
    raise RuntimeError("docker daemon unreachable")
RuntimeError: docker daemon unreachable


check failed for reach 10
Traceback (most recent call last):
  File "/home/ert/dev/git-repos/fim/2d-fim/twod-fim-deployment/orchestrator/recon/check.py", line 270, in run_check
    ref = runner.submit(decision.step, PAYLOADS[decision.step](reach_id))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3266158/4023097037.py", line 3, in submit
    raise RuntimeError("docker daemon unreachable")
RuntimeError: docker daemon unreachable


check failed for reach 10
Traceback (most recent call last):
  File "/home/ert/dev/git-repos/fim/2d-fim/twod-fim-deployment/orchestrator/recon/check.py", line 270, in run_check
    ref = runner.submit(decision.step, PAYLOADS[decision.step](reach_id))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3266158/4023097037.py", line 3, in submit
    raise RuntimeError("docker daemon unreachable")
RuntimeError: docker daemon unreachable


check failed for reach 10
Traceback (most recent call last):
  File "/home/ert/dev/git-repos/fim/2d-fim/twod-fim-deployment/orchestrator/recon/check.py", line 270, in run_check
    ref = runner.submit(decision.step, PAYLOADS[decision.step](reach_id))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3266158/4023097037.py", line 3, in submit
    raise RuntimeError("docker daemon unreachable")
RuntimeError: docker daemon unreachable


check failed for reach 10
Traceback (most recent call last):
  File "/home/ert/dev/git-repos/fim/2d-fim/twod-fim-deployment/orchestrator/recon/check.py", line 270, in run_check
    ref = runner.submit(decision.step, PAYLOADS[decision.step](reach_id))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3266158/4023097037.py", line 3, in submit
    raise RuntimeError("docker daemon unreachable")
RuntimeError: docker daemon unreachable


,attempt,consecutive_failures,halted,due
0,1,1,False,False
1,2,2,False,False
2,3,3,False,False
3,4,4,False,False
4,5,5,True,False
5,6,6,True,False


after clear_halt, due: [10]


## Case 11 — a job nobody can account for

**Setup:** a job in flight whose reference means nothing any more — container
reaped, batch history aged out.
**Expect:** left alone during a grace period; then the marker is cleared with
**no failure recorded**, because we do not know that it failed. The next check
asks storage, which is the only party with an answer. The worst case is a
duplicate submission, which content-addressing makes harmless.

In [12]:
runner = reset(11)
check.run_check(11, runner)
runner.status = JobStatus.UNKNOWN

print("still young:", jobs.status_pass(runner)[0]["action"])
with db.connect() as conn:
    conn.execute("UPDATE reach_processing SET current_step_started_at = now() - interval '30 min'"
                 " WHERE reach_id = 11")
print("past grace: ", jobs.status_pass(runner)[0]["action"])
print("failures:   ", db.one("SELECT consecutive_failures FROM reach_processing WHERE reach_id = 11"))
print("next check: ", check.run_check(11, runner))

still young: left alone
past grace:  lost track, marker cleared
failures:    {'consecutive_failures': 0}


next check:  reach 11 | rev 0 | RunStep | submitted build_model (stand-in-11-)


## Case 12 — a sweep settles

**Setup:** three fresh reaches (all terminal, so no dependencies here — the
cascade is `03_run_network.ipynb`'s story).
**Expect:** sweeps submit, then record, and once every reach is satisfied a
sweep does nothing at all. A loop that has caught up is quiet.

In [13]:
runner = reset(21, 22, 23)
rounds = []
for n in range(1, 6):
    if n == 2:
        runner.status = JobStatus.SUCCEEDED
        for rid in (21, 22, 23):
            put_model(rid)
    if n == 3:
        for rid in (21, 22, 23):
            put_nd_library(rid)
    jobs.status_pass(runner)
    results = check.sweep(runner)
    rounds.append({"sweep": n, "checked": len(results),
                   "decisions": ", ".join(sorted({r.decision for r in results})) or "-",
                   "submitted_total": len(runner.submitted)})
display(pd.DataFrame(rounds))
display(pd.DataFrame(db.query(
    "SELECT reach_id, state, model_id, nd_discharges FROM reach_status ORDER BY reach_id")))

,sweep,checked,decisions,submitted_total
0,1,3,RunStep,3
1,2,3,RunStep,6
2,3,3,NoGap,6
3,4,0,-,6
4,5,0,-,6


,reach_id,state,model_id,nd_discharges
0,21,finished,04e7b6e8_N10S10E10W10,3
1,22,finished,7db51e37_N10S10E10W10,3
2,23,finished,42365efe_N10S10E10W10,3


In [14]:
with db.connect() as conn:
    conn.execute("TRUNCATE reach_network, lakes CASCADE")
    conn.execute("DELETE FROM desired_state_defaults")
for rid in (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 21, 22, 23):
    clear_storage(rid)
print("cleaned up")

cleaned up


## What these cases establish

| | |
|---|---|
| **Existing work is adopted** | intent implies the address; whatever is there and verifies, counts |
| **Steps are proved one at a time** | a model does not make a reach finished; each rung is its own claim |
| **Work is never repeated** | the in-flight marker suppresses resubmission and cannot wedge |
| **Only storage is believed** | half-written models are invisible; lying manifests are refused |
| **Proof is all or nothing** | a library that does not span intent produces no row, not a partial one |
| **Undo is deletion** | removing the model removes both proofs, because runs are addressed beneath it |
| **Intent changes are ordinary** | the address moves, the gap reopens, reverting is free |
| **Identity and realization split the cost** | a solver upgrade redoes the runs and keeps the terrain |
| **Missing data is not failure** | a reach that drains nowhere awaits inputs, it is not retried |
| **Failure is bounded** | backoff, then halted, then a person |
| **Uncertainty is safe** | an unaccountable job costs at most one duplicate build |